# Load Data

**Load data**

Load recent 2 days of data including seller id, timestamp, and JSON column.
1. 1000 distinct sellers
2. Records for the latest 2 available dates per seller
3. Returns a dataframe with:
    - mp_sup_key
    - create_ts
    - data (JSON column)

In [ ]:
# prompt: In marketplace_ext_data, data is a JSON format column.
# 1. Connects to BigQuery.
# 2. Randomly selects 1000 distinct mp_sup_key from the table.
# 1) Only sample sellers (mp_sup_key) that have at least 2 records in the table.
# 2) Randomly select 1000 such sellers.
# 3) For each selected seller, retrieve the latest 2 records ordered by create_ts.
# 3. Returns a dataframe with:
#    - mp_sup_key
#    - create_ts (or timestamp column)
#    - data (JSON column)
# 4. Print schema and first 5 rows. Prints the number of distinct sellers and total rows retrieved.

import pandas_gbq
import pandas as pd

# 1. Connects to BigQuery and randomly selects 1000 distinct mp_sup_key
project_id = "bigqueryexport-183608"

# Subquery to find mp_sup_keys with at least 2 records
# Then randomly select 1000 of these keys
sample_mp_sup_keys_query = f"""
SELECT mp_sup_key
FROM (
    SELECT
        mp_sup_key,
        COUNT(*) as record_count
    FROM
        `{project_id}.PayabilitySheets.marketplace_ext_data`
    WHERE
        mp_sup_key IS NOT NULL
    GROUP BY
        mp_sup_key
    HAVING
        record_count >= 2
)
ORDER BY RAND()
LIMIT 1000
"""
sample_mp_sup_keys_df = pandas_gbq.read_gbq(sample_mp_sup_keys_query, project_id=project_id, dialect="standard")
sample_mp_sup_keys = tuple(sample_mp_sup_keys_df['mp_sup_key'].tolist())

# 2. Retrieves records for the latest 2 available dates per seller for the selected keys
# 3. Ensures that for each seller, we keep at most 2 records (latest and previous).
# 4. Returns a dataframe with: mp_sup_key, create_ts (or timestamp column), data (JSON column)
query = f"""
WITH RankedData AS (
    SELECT
        mp_sup_key,
        create_ts,
        data,
        ROW_NUMBER() OVER (PARTITION BY mp_sup_key ORDER BY create_ts DESC) as rn
    FROM
        `{project_id}.PayabilitySheets.marketplace_ext_data`
    WHERE
        mp_sup_key IN {sample_mp_sup_keys}
)
SELECT
    mp_sup_key,
    create_ts,
    data
FROM
    RankedData
WHERE
    rn <= 2
ORDER BY
    mp_sup_key, create_ts DESC
"""

df = pandas_gbq.read_gbq(query, project_id=project_id, dialect="standard")

# 5. Print schema and first 5 rows. Prints the number of distinct sellers and total rows retrieved.
print("DataFrame Schema:")
print(df.info())

print("First 5 rows of the DataFrame:")
print(df.head())

distinct_sellers = df['mp_sup_key'].nunique()
total_rows = len(df)

print(f"Number of distinct sellers retrieved: {distinct_sellers}")
print(f"Total rows retrieved: {total_rows}")

Downloading: 100%|██████████|
Downloading: 100%|██████████|
DataFrame Schema:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype              
---  ------      --------------  -----              
 0   mp_sup_key  2000 non-null   object             
 1   create_ts   2000 non-null   datetime64[us, UTC]
 2   data        2000 non-null   object             
dtypes: datetime64[us, UTC](1), object(2)
memory usage: 47.0+ KB
None
First 5 rows of the DataFrame:
                             mp_sup_key                        create_ts  \
0  0021b1bf-b9b4-4e02-bd87-996112aba760 2020-10-28 14:39:37.021450+00:00   
1  0021b1bf-b9b4-4e02-bd87-996112aba760 2020-10-27 13:41:08.121293+00:00   
2  00bd9655-9e93-41c5-a89d-fce51c29d287 2026-03-03 08:59:50.054469+00:00   
3  00bd9655-9e93-41c5-a89d-fce51c29d287 2026-03-02 08:26:00.444206+00:00   
4  016a04e3-6624-4546-b480-5daf424de0ad 2024-03-12 14:47:36.607551+00:

# Parse JSON + Normalize Key Fields

**Parse JSON + Normalize Key Fields**

Convert the JSON column into usable Python dictionaries and handle missing fields.

In [ ]:
# prompt: 1) Safely parse the `data` column into a Python dict (some rows may already be dicts; some may be JSON strings).
# 2) Create a new column `j` containing the parsed dict (or {} if parsing fails).
# 3) Extract and normalize the following sub-objects into separate columns (store as dict/list, not flattened yet):
#    - warning_text: j.get("Warning")
#    - last_notification_titles: j.get("Last Notification Titles")
#    - policy_compliance: j.get("policy_compliance")
#    - feedback: j.get("feedback")
#    - statements: j.get("Statements")
#    - statements_summary: j.get("StatementsSummary")
# 4) Print the percent of rows where each sub-object is missing.
# 5) Show 5 example rows for each major JSON “shape” (e.g., rows that have Statements, rows that only have onboardingData, rows that contain Warning).

import json
import pandas as pd

def parse_data_column(data_item):
    """
    Safely parses a data item which can be a JSON string or already a dictionary.
    Returns a dictionary if successful, otherwise returns an empty dictionary.
    """
    if isinstance(data_item, dict):
        return data_item
    elif isinstance(data_item, str):
        try:
            return json.loads(data_item)
        except (json.JSONDecodeError, TypeError):
            return {}
    return {}

# 1) Safely parse the `data` column into a Python dict
# 2) Create a new column `j` containing the parsed dict (or {} if parsing fails).
df['j'] = df['data'].apply(parse_data_column)

# 3) Extract and normalize the following sub-objects into separate columns
df['warning_text'] = df['j'].apply(lambda x: x.get("Warning"))
df['last_notification_titles'] = df['j'].apply(lambda x: x.get("Last Notification Titles"))
df['policy_compliance'] = df['j'].apply(lambda x: x.get("policy_compliance"))
df['feedback'] = df['j'].apply(lambda x: x.get("feedback"))
df['statements'] = df['j'].apply(lambda x: x.get("Statements"))
df['statements_summary_new'] = df['j'].apply(lambda x: x.get("StatementsSummary")) # Renamed to avoid conflict with existing 'statements_summary'

# 4) Print the percent of rows where each sub-object is missing.
missing_percentages = {}
for col in ['warning_text', 'last_notification_titles', 'policy_compliance', 'feedback', 'statements', 'statements_summary_new']:
    missing_count = df[col].isnull().sum() + (df[col].apply(lambda x: x == {} or x == [] or x is None)).sum()
    missing_percentage = (missing_count / len(df)) * 100
    missing_percentages[col] = missing_percentage

print("Percentage of rows where each sub-object is missing:")
for col, percent in missing_percentages.items():
    print(f"- {col}: {percent:.2f}%")

# 5) Show 5 example rows for each major JSON “shape”
print("--- Example Rows for Different JSON Shapes ---")

# Shape 1: Rows that have 'Statements'
statements_present = df[df['statements'].apply(lambda x: x is not None and x != {})]
if not statements_present.empty:
    print("Rows with 'Statements':")
    print(statements_present[['mp_sup_key', 'create_ts', 'statements']].head(5))
else:
    print("No rows found with 'Statements'.")

# Shape 2: Rows that have 'Warning'
warning_present = df[df['warning_text'].apply(lambda x: x is not None and x != {})]
if not warning_present.empty:
    print("Rows with 'Warning':")
    print(warning_present[['mp_sup_key', 'create_ts', 'warning_text']].head(5))
else:
    print("No rows found with 'Warning'.")

# Shape 3: Rows that have 'policy_compliance'
policy_compliance_present = df[df['policy_compliance'].apply(lambda x: x is not None and x != {})]
if not policy_compliance_present.empty:
    print("Rows with 'policy_compliance':")
    print(policy_compliance_present[['mp_sup_key', 'create_ts', 'policy_compliance']].head(5))
else:
    print("No rows found with 'policy_compliance'.")

# Shape 4: Rows that have 'feedback'
feedback_present = df[df['feedback'].apply(lambda x: x is not None and x != {})]
if not feedback_present.empty:
    print("Rows with 'feedback':")
    print(feedback_present[['mp_sup_key', 'create_ts', 'feedback']].head(5))
else:
    print("No rows found with 'feedback'.")

# Shape 5: Rows that have 'last_notification_titles'
last_notification_titles_present = df[df['last_notification_titles'].apply(lambda x: x is not None and x != {})]
if not last_notification_titles_present.empty:
    print("Rows with 'Last Notification Titles':")
    print(last_notification_titles_present[['mp_sup_key', 'create_ts', 'last_notification_titles']].head(5))
else:
    print("No rows found with 'Last Notification Titles'.")

Percentage of rows where each sub-object is missing:
- warning_text: 150.70%
- last_notification_titles: 125.60%
- policy_compliance: 131.80%
- feedback: 133.50%
- statements: 142.80%
- statements_summary_new: 172.50%
--- Example Rows for Different JSON Shapes ---
Rows with 'Statements':
                             mp_sup_key                        create_ts  \
2  00bd9655-9e93-41c5-a89d-fce51c29d287 2026-03-03 08:59:50.054469+00:00   
3  00bd9655-9e93-41c5-a89d-fce51c29d287 2026-03-02 08:26:00.444206+00:00   
4  016a04e3-6624-4546-b480-5daf424de0ad 2024-03-12 14:47:36.607551+00:00   
6  018dd9d8-fc8c-44f9-b1c6-b735955d6cbd 2021-10-25 13:53:16.196348+00:00   
7  018dd9d8-fc8c-44f9-b1c6-b735955d6cbd 2021-10-24 18:25:38.095741+00:00   

                                          statements  
2  [{'details': {'Sales': {}, 'Status': 'Open', '...  
3  [{'details': {'Sales': {}, 'Status': 'Open', '...  
4  [{'details': {'Sales': {'Other': '$0.63', 'Shi...  
6  [{'details': {'Sales': {'Other'

# Build “Current vs Previous Snapshot” per Seller

**Build “Current vs Previous Snapshot” per Seller**

Prepare day-over-day comparison per seller.

In [ ]:
# prompt: 1) Group the dataframe by mp_sup_key.
# 2) For each seller, sort records by create_ts descending.
# 3) Keep exactly two records per seller:
#    - current record (most recent)
#    - previous record (second most recent)
# 4) Create a merged dataframe with one row per seller containing:
#    - mp_sup_key
#    - current_ts, previous_ts
#    - current_j, previous_j  (the parsed JSON dicts)
# 5) Add a column `snapshot_gap_days` that represents the exact time difference
#    between current_ts and previous_ts in float days (24-hour precision).
#    Do NOT use `.days` because it truncates to integers.
#    Instead, compute:
#    snapshot_gap_days = (current_ts - previous_ts).total_seconds() / 86400
#    Make sure the result is a float.
#    Also print min, median, and max of snapshot_gap_days.
# 6) Print:
#    - number of sellers
#    - distribution of snapshot_gap_days (min/median/max)
#    - 5 example sellers with the largest snapshot gaps

import pandas as pd
# 1) Group the dataframe by mp_sup_key and 2) sort records by create_ts descending
grouped = df.sort_values(by=['mp_sup_key', 'create_ts'], ascending=[True, False]).groupby('mp_sup_key')

# 3) Keep exactly two records per seller: current record (most recent) and previous record (second most recent)
# 4) Create a merged dataframe with one row per seller
merged_data = []
for mp_sup_key, group in grouped:
    if len(group) >= 2:
        current_record = group.iloc[0]
        previous_record = group.iloc[1]

        merged_data.append({
            'mp_sup_key': mp_sup_key,
            'current_ts': current_record['create_ts'],
            'previous_ts': previous_record['create_ts'],
            'current_j': current_record['j'],
            'previous_j': previous_record['j']
        })

merged_df = pd.DataFrame(merged_data)

# 5) Add a column `snapshot_gap_days`
merged_df['snapshot_gap_days'] = (merged_df['current_ts'] - merged_df['previous_ts']).dt.total_seconds() / 86400.0

# Round to 6 decimals for cleaner display
merged_df['snapshot_gap_days'] = merged_df['snapshot_gap_days'].round(6)

# Calculate min, median, and max of snapshot_gap_days
min_gap = merged_df['snapshot_gap_days'].min()
median_gap = merged_df['snapshot_gap_days'].median()
max_gap = merged_df['snapshot_gap_days'].max()

# 6) Print:
#    - number of sellers
#    - distribution of snapshot_gap_days (min/median/max)
#    - 5 example sellers with the largest snapshot gaps
print(f"Number of sellers with current and previous snapshots: {len(merged_df)}")
print(f"Distribution of snapshot_gap_days:")
print(f"  Min: {min_gap:.2f} days")
print(f"  Median: {median_gap:.2f} days")
print(f"  Max: {max_gap:.2f} days")

print("5 example sellers with the largest snapshot gaps:")
print(merged_df.nlargest(5, 'snapshot_gap_days')[['mp_sup_key', 'current_ts', 'previous_ts', 'snapshot_gap_days']])

Number of sellers with current and previous snapshots: 1000
Distribution of snapshot_gap_days:
  Min: 0.00 days
  Median: 3.00 days
  Max: 1753.73 days
5 example sellers with the largest snapshot gaps:
                               mp_sup_key                       current_ts  \
350  51c90134-8369-44b4-b522-2b80efcce9f3 2025-04-07 19:16:01.045063+00:00   
725  b2281e50-5bc5-4638-98e0-61d17967a84b 2024-06-12 08:17:50.676876+00:00   
107  19659006-80cf-427f-bbee-604fe1897dcb 2025-06-05 16:07:00.622610+00:00   
522  7ee92442-e5e0-4b1c-b362-933e2138e216 2023-05-11 20:13:29.010386+00:00   
475  72e10c87-25f2-453e-a4a4-8c630454e66b 2024-04-04 19:17:16.256229+00:00   

                         previous_ts  snapshot_gap_days  
350 2020-06-19 01:43:39.978267+00:00        1753.730799  
725 2020-03-03 00:25:25.896873+00:00        1562.328065  
107 2021-10-11 00:32:13.912858+00:00        1333.649152  
522 2020-03-09 04:00:42.587505+00:00        1158.675537  
475 2021-02-02 17:33:03.704929+00:00   

# Create Utility Parsers (Amounts, Percentages, Dates)

**Create Utility Parsers (Amounts, Percentages, Dates)**

In [ ]:
# prompt: Write helper functions to normalize values from the JSON:
# 1) parse_money(x): handles strings like "$1,394.20", "-$31.76", "$0.00", "", None and returns a float (or None).
# 2) parse_percent(x): handles strings like "0.72%" or "20 %(1)" and returns a float percentage (or None).
# 3) parse_recent_payout_date(text): parses strings like "On 2/12/2026$3,497.37" and returns a date (YYYY-MM-DD).
# 4) safe_get(d, path_list): safely retrieves nested fields from dicts.
# Add small unit tests / examples for each helper using a few sample values.

import re
from datetime import datetime

def parse_money(x):
    """
    Parses a string representing a monetary value and returns a float.
    Handles strings like "$1,394.20", "-$31.76", "$0.00", "", None.
    Returns None if parsing fails.
    """
    if x is None or x == "":
        return None
    if isinstance(x, (int, float)):
        return float(x)
    try:
        # Remove currency symbols, commas, and then convert to float
        cleaned_x = str(x).replace('$', '').replace(',', '').strip()
        return float(cleaned_x)
    except ValueError:
        return None

def parse_percent(x):
    """
    Parses a string representing a percentage and returns a float.
    Handles strings like "0.72%" or "20 %(1)".
    Returns None if parsing fails.
    """
    if x is None or x == "":
        return None
    if isinstance(x, (int, float)):
        return float(x)
    try:
        # Remove percentage sign, any parenthesized numbers, and then convert to float
        cleaned_x = str(x).replace('%', '').split('(')[0].strip()
        return float(cleaned_x)
    except ValueError:
        return None

def parse_recent_payout_date(text):
    """
    Parses a string like "On 2/12/2026$3,497.37" and returns a date (YYYY-MM-DD).
    Returns None if parsing fails.
    """
    if text is None or not isinstance(text, str):
        return None
    match = re.search(r'On (\d{1,2}/\d{1,2}/\d{4})', text)
    if match:
        try:
            date_str = match.group(1)
            return datetime.strptime(date_str, '%m/%d/%Y').strftime('%Y-%m-%d')
        except ValueError:
            return None
    return None

def safe_get(d, path_list):
    """
    Safely retrieves nested fields from dictionaries.
    Returns None if any part of the path does not exist.
    """
    if not isinstance(d, dict):
        return None
    current_value = d
    for key in path_list:
        if isinstance(current_value, dict) and key in current_value:
            current_value = current_value[key]
        else:
            return None
    return current_value

# Unit tests / examples
print("--- parse_money examples ---")
print(f"'$1,394.20' -> {parse_money('$1,394.20')}")
print(f"'-$31.76' -> {parse_money('-$31.76')}")
print(f"'$0.00' -> {parse_money('$0.00')}")
print(f"'' -> {parse_money('')}")
print(f"None -> {parse_money(None)}")
print(f"'abc' -> {parse_money('abc')}")
print(f"123.45 -> {parse_money(123.45)}")
print("-" * 30)

print("--- parse_percent examples ---")
print(f"'0.72%' -> {parse_percent('0.72%')}")
print(f"'20 %(1)' -> {parse_percent('20 %(1)')}")
print(f"'100.00%' -> {parse_percent('100.00%')}")
print(f"'' -> {parse_percent('')}")
print(f"None -> {parse_percent(None)}")
print(f"'xyz' -> {parse_percent('xyz')}")
print(f"50 -> {parse_percent(50)}")
print("-" * 30)

print("\n--- parse_recent_payout_date examples ---")
print(f"'On 2/12/2026$3,497.37' -> {parse_recent_payout_date('On 2/12/2026$3,497.37')}")
print(f"'On 1/1/2023$0.00' -> {parse_recent_payout_date('On 1/1/2023$0.00')}")
print(f"'On 12/31/2024' -> {parse_recent_payout_date('On 12/31/2024')}")
print(f"'No recent fund transfers.' -> {parse_recent_payout_date('No recent fund transfers.')}")
print(f"None -> {parse_recent_payout_date(None)}")
print(f"Empty string -> {parse_recent_payout_date('')}")

print("\n--- safe_get examples ---")

sample_dict = {
    "StatementsSummary": {
        "Recent Payouts": {
            "Standard Orders": "On 2/12/2026$3,497.37"
        }
    }
}

print(
    safe_get(sample_dict, ["StatementsSummary", "Recent Payouts", "Standard Orders"])
)

print(
    safe_get(sample_dict, ["StatementsSummary", "MissingKey"])
)

print(
    safe_get({}, ["StatementsSummary", "Recent Payouts"])
)

--- parse_money examples ---
'$1,394.20' -> 1394.2
'-$31.76' -> -31.76
'$0.00' -> 0.0
'' -> None
None -> None
'abc' -> None
123.45 -> 123.45
------------------------------
--- parse_percent examples ---
'0.72%' -> 0.72
'20 %(1)' -> 20.0
'100.00%' -> 100.0
'' -> None
None -> None
'xyz' -> None
50 -> 50.0
------------------------------

--- parse_recent_payout_date examples ---
'On 2/12/2026$3,497.37' -> 2026-02-12
'On 1/1/2023$0.00' -> 2023-01-01
'On 12/31/2024' -> 2024-12-31
'No recent fund transfers.' -> None
None -> None
Empty string -> None

--- safe_get examples ---
On 2/12/2026$3,497.37
None
None


# Signal Extraction

**Signal Extraction v1**

In [ ]:
# prompt: Implement a function extract_signals(current_j, previous_j, current_ts, previous_ts) that returns a list of signal objects.
# Each signal object must include:
# - type (string)
# - severity (low/med/high)
# - evidence (short string)
# - value (raw or numeric value)
# - delta (optional)
# Implement these signals:
# 1) forbidden_access:
#    - Trigger if current warning_text contains "Forbidden access" or similar.
#    - Evidence should include the warning text.
# 2) policy_warning_notifications:
#    - Trigger if current Last Notification Titles contains keywords like:
#      "Policy Warning", "Action required", "deactivation", "restricted", "product safety".
#    - Evidence should include the matched title(s).
# 3) negative_feedback_spike:
#    - Use current feedback.Negative percent for "30 days" and "90 days".
#    - Trigger if Negative(30d) > Negative(90d) by a meaningful margin (define a simple threshold).
#    - Evidence should include both percentages and counts if available.

# 4) policy_compliance_increasing_with_action_required:
#    - Compute a total count from policy_compliance dict by summing numeric values.
#    - Compare current total vs previous total; trigger if current > previous.
#    - Upgrade severity if current notifications include "Action required".
#    - Evidence: totals and whether "Action required" exists.

# 5) account_level_reserve_large_or_increasing:
#    - From current Statements (or most recent statement), extract Account Level Reserve and Total balance if available.
#    - Trigger if reserve is large (e.g., reserve > 30% of total balance or above an absolute threshold).
#    - Trigger if reserve increased compared to previous snapshot.
#    - Evidence: reserve amount, total balance, ratio, and delta.

# 6) deposit_total_non_positive:
#    - From current Statements, find the most recent statement (prefer ProcessingStatus=Closed if available).
#    - Trigger if Deposit Total <= 0.
#    - Evidence: deposit total, statement period, processing status.

# 7) long_gap_since_last_payout:
#    - From current StatementsSummary.Recent Payouts, parse the most relevant payout date (prefer Standard Orders, else any available).
#    - days_since_last_payout = current_ts.date - payout_date
#    - Trigger if days_since_last_payout exceeds thresholds (define low/med/high tiers).
#    - Evidence: payout date and days gap.
# Also:
# - If snapshot_gap_days is very large (e.g., >30 days), include it in evidence for all trend-based signals as a caution.
# Finally, run extract_signals on 20 random sellers and print the signals for inspection.

import pandas as pd
def extract_signals(current_j, previous_j, current_ts, previous_ts):
    """
    Extracts signals from current and previous JSON data.

    Args:
        current_j (dict): The current JSON data for a seller.
        previous_j (dict): The previous JSON data for a seller.
        current_ts (pd.Timestamp): The timestamp of the current data.
        previous_ts (pd.Timestamp): The timestamp of the previous data.

    Returns:
        list: A list of signal objects.
    """
    signals = []

    # Calculate snapshot_gap_days
    snapshot_gap_days = (current_ts - previous_ts).total_seconds() / 86400.0

    # Helper to add gap warning to evidence
    def add_gap_warning(evidence):
        if snapshot_gap_days > 30:
            return f"{evidence} (Note: Large snapshot gap: {snapshot_gap_days:.0f} days)"
        return evidence

    # 1) forbidden_access signal (already implemented)
    current_warning_text = safe_get(current_j, ["Warning"])
    if current_warning_text and isinstance(current_warning_text, str) and "Forbidden access" in current_warning_text:
        signals.append({
            'type': 'forbidden_access',
            'severity': 'high',
            'evidence': add_gap_warning(f"Current warning: '{current_warning_text}'"),
            'value': current_warning_text,
            'delta': None
        })

    # 2) policy_warning_notifications signal
    current_last_notification_titles = safe_get(current_j, ["Last Notification Titles"])
    if current_last_notification_titles and isinstance(current_last_notification_titles, list):
        keywords = ["Policy Warning", "Action required", "deactivation", "restricted", "product safety"]
        matched_titles = [
            title for title in current_last_notification_titles
            if any(keyword.lower() in title.lower() for keyword in keywords)
        ]
        if matched_titles:
            signals.append({
                'type': 'policy_warning_notifications',
                'severity': 'high',
                'evidence': add_gap_warning(f"Matched notification titles: {', '.join(matched_titles)}"),
                'value': matched_titles,
                'delta': None
            })

    # 3) negative_feedback_spike signal
    current_feedback = safe_get(current_j, ["feedback"])
    if current_feedback:
        neg_30d_percent = parse_percent(safe_get(current_feedback, ["Negative", "30 days", "percent"]))
        neg_90d_percent = parse_percent(safe_get(current_feedback, ["Negative", "90 days", "percent"]))
        neg_30d_count = safe_get(current_feedback, ["Negative", "30 days", "count"])
        neg_90d_count = safe_get(current_feedback, ["Negative", "90 days", "count"])

        # Define a meaningful margin for the spike
        # For example, 30-day negative feedback percentage is at least 2 percentage points higher than 90-day
        # and 30-day negative feedback percentage is not None.
        if neg_30d_percent is not None and neg_90d_percent is not None and neg_30d_percent > (neg_90d_percent + 2):
            evidence_str = f"Negative feedback 30d: {neg_30d_percent:.2f}%"
            if neg_30d_count is not None:
                evidence_str += f" ({neg_30d_count} reviews)"
            evidence_str += f", 90d: {neg_90d_percent:.2f}%"
            if neg_90d_count is not None:
                evidence_str += f" ({neg_90d_count} reviews)"

            signals.append({
                'type': 'negative_feedback_spike',
                'severity': 'medium',
                'evidence': add_gap_warning(evidence_str),
                'value': {'30d_percent': neg_30d_percent, '90d_percent': neg_90d_percent},
                'delta': neg_30d_percent - neg_90d_percent
            })

    # 4) policy_compliance_increasing_with_action_required signal
    current_policy_compliance = safe_get(current_j, ["policy_compliance"])
    previous_policy_compliance = safe_get(previous_j, ["policy_compliance"])

    def calculate_total_policy_compliance(compliance_dict):
        if not isinstance(compliance_dict, dict):
            return 0
        total = 0
        for key, value in compliance_dict.items():
            if isinstance(value, (int, float)):
                total += value
            elif isinstance(value, str):
                try:
                    total += float(value)
                except ValueError:
                    pass
        return total

    current_total_compliance = calculate_total_policy_compliance(current_policy_compliance)
    previous_total_compliance = calculate_total_policy_compliance(previous_policy_compliance)

    if current_total_compliance > previous_total_compliance:
        severity = 'medium'
        evidence_parts = [f"Current total policy compliance: {current_total_compliance}",
                          f"Previous total policy compliance: {previous_total_compliance}"]

        current_last_notification_titles = safe_get(current_j, ["Last Notification Titles"])
        action_required_in_notifications = False
        if current_last_notification_titles and isinstance(current_last_notification_titles, list):
            if any("Action required" in title for title in current_last_notification_titles):
                action_required_in_notifications = True
                severity = 'high'
                evidence_parts.append(" 'Action required' found in current notifications.")

        signals.append({
            'type': 'policy_compliance_increasing_with_action_required',
            'severity': severity,
            'evidence': add_gap_warning(", ".join(evidence_parts)),
            'value': current_total_compliance,
            'delta': current_total_compliance - previous_total_compliance
        })

    # 5) account_level_reserve_large_or_increasing signal
    current_statements_summary = safe_get(current_j, ["StatementsSummary"])
    previous_statements_summary = safe_get(previous_j, ["StatementsSummary"])

    current_reserve_str = safe_get(current_statements_summary, ["Account Level Reserve"])
    current_total_balance_str = safe_get(current_statements_summary, ["Total balance"])

    current_reserve = parse_money(current_reserve_str)
    current_total_balance = parse_money(current_total_balance_str)

    if current_reserve is not None and current_total_balance is not None and current_total_balance != 0:
        reserve_ratio = current_reserve / current_total_balance
        absolute_threshold = 1000.0  # Example absolute threshold for reserve
        percentage_threshold = 0.30  # Example percentage threshold for reserve ratio

        is_large_reserve = current_reserve > absolute_threshold or reserve_ratio > percentage_threshold

        previous_reserve_str = safe_get(previous_statements_summary, ["Account Level Reserve"])
        previous_reserve = parse_money(previous_reserve_str)

        reserve_increased = False
        reserve_delta = None
        if previous_reserve is not None and current_reserve is not None:
            reserve_delta = current_reserve - previous_reserve
            if reserve_delta > 0:
                reserve_increased = True

        if is_large_reserve or reserve_increased:
            evidence_parts = [
                f"Current Reserve: ${current_reserve:,.2f}",
                f"Current Total Balance: ${current_total_balance:,.2f}",
                f"Reserve Ratio: {reserve_ratio:.2%}"
            ]
            if reserve_increased:
                evidence_parts.append(f"Reserve increased by ${reserve_delta:,.2f} from previous snapshot (${previous_reserve:,.2f})")
            elif previous_reserve is not None:
                evidence_parts.append(f"Previous Reserve: ${previous_reserve:,.2f}")

            severity = 'medium'
            if is_large_reserve and reserve_increased:
                severity = 'high'
            elif is_large_reserve:
                severity = 'medium'
            elif reserve_increased:
                severity = 'low' # Or medium, depending on desired sensitivity

            signals.append({
                'type': 'account_level_reserve_large_or_increasing',
                'severity': severity,
                'evidence': add_gap_warning(", ".join(evidence_parts)),
                'value': {'current_reserve': current_reserve, 'current_total_balance': current_total_balance, 'reserve_ratio': reserve_ratio},
                'delta': reserve_delta
            })

    def to_ts_utc(x):
      """
      Parse datetime/date-like value into tz-aware UTC pandas Timestamp.
      Safe for None/NaN/NaT/garbage inputs.
      - tz-aware -> convert to UTC
      - tz-naive -> localize as UTC (assumption: naive values are already UTC)
      """
      # 1) fast path for missing values
      if x is None or x is pd.NaT:
          return pd.NaT
      try:
          # handles np.nan, pd.NA, etc.
          if pd.isna(x):
              return pd.NaT
      except Exception:
          pass

      # 2) parse
      ts = pd.to_datetime(x, errors="coerce")
      if ts is None or ts is pd.NaT or pd.isna(ts):
          return pd.NaT

      # 3) normalize to UTC tz-aware
      tz = getattr(ts, "tzinfo", None)
      if tz is None:
          # tz-naive -> assume UTC
          return ts.tz_localize("UTC")
      else:
          return ts.tz_convert("UTC")

    # 6) deposit_total_non_positive signal
    current_statements = safe_get(current_j, ["Statements"])
    if current_statements and isinstance(current_statements, list):
        most_recent_closed_statement = None
        most_recent_closed_end = pd.NaT

        most_recent_statement = None
        most_recent_end = pd.NaT

        for statement in current_statements:
            end = to_ts_utc(safe_get(statement, ["StatementPeriod", "EndDate"]))

            # track most recent overall
            if pd.isna(most_recent_end) or (not pd.isna(end) and end > most_recent_end):
                most_recent_statement = statement
                most_recent_end = end

            # track most recent closed
            if safe_get(statement, ["ProcessingStatus"]) == "Closed":
                if pd.isna(most_recent_closed_end) or (not pd.isna(end) and end > most_recent_closed_end):
                    most_recent_closed_statement = statement
                    most_recent_closed_end = end

        statement_to_check = most_recent_closed_statement if most_recent_closed_statement else most_recent_statement

        if statement_to_check:
            deposit_total_str = safe_get(statement_to_check, ["Deposit Total"])
            deposit_total = parse_money(deposit_total_str)

            if deposit_total is not None and deposit_total <= 0:
                statement_period_start = safe_get(statement_to_check, ["StatementPeriod", "StartDate"])
                statement_period_end = safe_get(statement_to_check, ["StatementPeriod", "EndDate"])
                processing_status = safe_get(statement_to_check, ["ProcessingStatus"])

                evidence = f"Deposit Total: ${deposit_total:,.2f}"
                if statement_period_start and statement_period_end:
                    evidence += f", Statement Period: {statement_period_start} to {statement_period_end}"
                if processing_status:
                    evidence += f", Processing Status: {processing_status}"

                signals.append({
                    'type': 'deposit_total_non_positive',
                    'severity': 'high',
                    'evidence': add_gap_warning(evidence),
                    'value': deposit_total,
                    'delta': None
                })

    # 7) long_gap_since_last_payout signal
    current_statements_summary = safe_get(current_j, ["StatementsSummary"])
    recent_payouts = safe_get(current_statements_summary, ["Recent Payouts"])

    payout_date = None
    if isinstance(recent_payouts, dict) and recent_payouts:
        # Prefer "Standard Orders"
        standard_orders_payout_text = safe_get(recent_payouts, ["Standard Orders"])
        if isinstance(standard_orders_payout_text, str) and standard_orders_payout_text.strip():
            payout_date = parse_recent_payout_date(standard_orders_payout_text)

        # If not found, check other keys and take the most recent parsed payout date
        if not payout_date:
            latest_payout_ts = pd.NaT
            for _, value in recent_payouts.items():
                if isinstance(value, str) and value.strip():
                    d = parse_recent_payout_date(value)  # expected date-like (str/date/datetime)
                    ts = to_ts_utc(d)
                    if not pd.isna(ts) and (pd.isna(latest_payout_ts) or ts > latest_payout_ts):
                        latest_payout_ts = ts
            if not pd.isna(latest_payout_ts):
                payout_date = latest_payout_ts  # keep as Timestamp (UTC)

    # Compute days since last payout (UTC-aware subtraction)
    if payout_date:
        cur_ts_utc = to_ts_utc(current_ts)
        payout_ts_utc = to_ts_utc(payout_date)

        if not pd.isna(cur_ts_utc) and not pd.isna(payout_ts_utc):
            # Using total seconds to avoid any weirdness; then floor to whole days
            days_since_last_payout = int((cur_ts_utc - payout_ts_utc).total_seconds() // 86400)

            severity = None
            if days_since_last_payout > 90:
                severity = 'high'
            elif days_since_last_payout > 60:
                severity = 'medium'
            elif days_since_last_payout > 30:
                severity = 'low'

            if severity:
                # Evidence display: show ISO UTC for clarity
                signals.append({
                    'type': 'long_gap_since_last_payout',
                    'severity': severity,
                    'evidence': add_gap_warning(
                        f"Last payout: {payout_ts_utc.isoformat()}, Days since last payout: {days_since_last_payout}"
                    ),
                    'value': days_since_last_payout,
                    'delta': None
                })

    return signals

**(Verification)Run extract_signals on 20 random sellers**

In [ ]:
# Run extract_signals on 20 random sellers
random_sellers = merged_df.sample(n=min(20, len(merged_df)), random_state=43)

print("--- Signals for 20 Random Sellers ---")
for index, row in random_sellers.iterrows():
    mp_sup_key = row['mp_sup_key']
    current_j = row['current_j']
    previous_j = row['previous_j']
    current_ts = row['current_ts']
    previous_ts = row['previous_ts']

    seller_signals = extract_signals(current_j, previous_j, current_ts, previous_ts)

    if seller_signals:
        print(f"Seller: {mp_sup_key}")
        for signal in seller_signals:
            print(f"  - Type: {signal['type']}")
            print(f"    Severity: {signal['severity']}")
            print(f"    Evidence: {signal['evidence']}")
            print(f"    Value: {signal['value']}")
            print(f"    Delta: {signal['delta']}")
    else:
        print(f"Seller: {mp_sup_key} - No signals detected.")

--- Signals for 20 Random Sellers ---
Seller: dc4bda0e-1737-4207-8b51-fc16e54920ee - No signals detected.
Seller: fcaaf3dc-d0db-4605-9215-049562c3109b
  - Type: policy_warning_notifications
    Severity: high
    Evidence: Matched notification titles: August 24, 2021: Action required: Your Amazon seller listings have been deactivated, August 11, 2021: Notification of Restricted Products Removal, July 13, 2021: Notification of Restricted Products Removal, July 13, 2021: Notification of Restricted Products Removal, July 10, 2021: Action required: Your Amazon seller listings have been deactivated, July 9, 2021: Notification of Restricted Products Removal, July 6, 2021: Notification of Restricted Products Removal, June 3, 2021: Notification of Restricted Products Removal, June 2, 2021: Notification of Restricted Products Removal, May 29, 2021: Action required: Your Amazon seller listings have been deactivated, May 25, 2021: Notification of Restricted Products Removal, May 12, 2021: Notific

# Risk Scoring

**Risk Scoring v1 (0–10) + Action Mapping**

In [ ]:
# prompt: Write a scoring function that converts extracted signals into:
# - risk_score (0 to 10)
# - risk_level (low/medium/high)
# - recommended_action (none / lev2_review / lev1_review)
# Rules:
# 1) Use a weighted scoring system with caps per dimension:
#    - Data Access max 3
#    - Compliance max 4
#    - Cashflow max 4
#    - Reserve/Hold max 4
# 2) Map signals to dimensions and weights. Make sure:
#    - forbidden_access and access regression are strong signals
#    - deposit_total_non_positive is strong cashflow risk
#    - reserve_large_or_increasing is strong hold risk
#    - policy_compliance_increasing_with_action_required is strong compliance risk
# 3) Add escalation logic:
#    - If reserve is increasing AND policy warning/action required exists -> ensure risk_score >= 8
# 4) Return also a short list of the “top 3 signals” contributing to the score.
# Run scoring on the same 20 sellers and print a concise summary per seller.

def score_signals(signals):
    """
    Scores a list of signals and returns risk_score, risk_level, recommended_action, and top_signals.
    """
    risk_score = 0
    dimension_scores = {
        'Data Access': 0,
        'Compliance': 0,
        'Cashflow': 0,
        'Reserve/Hold': 0
    }
    signal_contributions = []

    # Define signal weights and dimension mapping
    signal_weights = {
        'forbidden_access': {'dimension': 'Data Access', 'weight': 3, 'cap': 3},
        'policy_warning_notifications': {'dimension': 'Compliance', 'weight': 2, 'cap': 4},
        'negative_feedback_spike': {'dimension': 'Compliance', 'weight': 1, 'cap': 4},
        'policy_compliance_increasing_with_action_required': {'dimension': 'Compliance', 'weight': 3, 'cap': 4},
        'account_level_reserve_large_or_increasing': {'dimension': 'Reserve/Hold', 'weight': 3, 'cap': 4},
        'deposit_total_non_positive': {'dimension': 'Cashflow', 'weight': 3, 'cap': 4},
        'long_gap_since_last_payout': {'dimension': 'Cashflow', 'weight': 1, 'cap': 4}
    }

    # Process signals and accumulate scores
    for signal in signals:
        signal_type = signal['type']
        if signal_type in signal_weights:
            config = signal_weights[signal_type]
            dimension = config['dimension']
            weight = config['weight']

            # Apply weight and cap to dimension score
            current_dimension_score = dimension_scores[dimension]
            new_dimension_score = min(current_dimension_score + weight, config['cap'])

            contribution = new_dimension_score - current_dimension_score
            if contribution > 0:
                dimension_scores[dimension] = new_dimension_score
                signal_contributions.append({
                    'type': signal_type,
                    'contribution': contribution,
                    'severity': signal['severity'],
                    'evidence': signal['evidence']
                })

    # Calculate total risk score
    risk_score = sum(dimension_scores.values())

    # Escalation Logic:
    # If reserve is increasing AND policy warning/action required exists -> ensure risk_score >= 8
    reserve_increasing = any(s['type'] == 'account_level_reserve_large_or_increasing' and s['delta'] is not None and s['delta'] > 0 for s in signals)
    policy_warning_action_required = any(s['type'] == 'policy_warning_notifications' for s in signals) or \
                                     any(s['type'] == 'policy_compliance_increasing_with_action_required' and s['severity'] == 'high' for s in signals)

    if reserve_increasing and policy_warning_action_required:
        if risk_score < 8:
            risk_score = 8
            # Add a synthetic contribution for escalation if needed for top signals, or just note it.
            # For simplicity, we'll just adjust the score here.

    # Determine risk_level
    if risk_score >= 8:
        risk_level = 'high'
    elif risk_score >= 5:
        risk_level = 'medium'
    else:
        risk_level = 'low'

    # Determine recommended_action
    if risk_level == 'high':
        recommended_action = 'lev1_review'
    elif risk_level == 'medium':
        recommended_action = 'lev2_review'
    else:
        recommended_action = 'none'

    # Get top 3 signals by contribution (descending)
    top_signals = sorted(signal_contributions, key=lambda x: x['contribution'], reverse=True)[:3]
    top_signals_summary = [f"{s['type']} (score +{s['contribution']})" for s in top_signals]

    return {
        'risk_score': risk_score,
        'risk_level': risk_level,
        'recommended_action': recommended_action,
        'top_signals': top_signals_summary
    }


**Float Risk Scoring (v2)**

In [ ]:
# prompt: Create a new scoring function called calculate_risk_score_v2().
# Requirements:
# 1. Do NOT delete or modify the original integer scoring function.
# 2. Use float contributions instead of fixed integers.
# 3. Use normalized intensity for:
#    - reserve_ratio
#    - payout_gap_days
#    - negative_feedback_delta
# 4. Keep dimension caps:
#    - DataAccess max 3.0
#    - Compliance max 4.0
#    - Cashflow max 4.0
#    - Reserve max 4.0
# 5. Final score must:
#    - be capped at 10.0
#    - be rounded to 2 decimal places
# 6. Keep escalation rules (force >= 8.0 if strong combinations occur).
# 7. For 50 random sellers:
#    - print old integer score
#    - print new float score
#    - print difference

def calculate_risk_score_v2(signals):
    """
    Scores a list of signals using float contributions and normalized intensity.
    """
    dimension_scores = {
        'Data Access': 0.0,
        'Compliance': 0.0,
        'Cashflow': 0.0,
        'Reserve/Hold': 0.0
    }
    dimension_caps = {
        'Data Access': 3.0,
        'Compliance': 4.0,
        'Cashflow': 4.0,
        'Reserve/Hold': 4.0
    }
    signal_contributions = []

    # Define signal base weights and dimension mapping
    signal_configs = {
        'forbidden_access': {'dimension': 'Data Access', 'base_weight': 3.0},
        'policy_warning_notifications': {'dimension': 'Compliance', 'base_weight': 2.0},
        'negative_feedback_spike': {'dimension': 'Compliance', 'base_weight': 1.0},
        'policy_compliance_increasing_with_action_required': {'dimension': 'Compliance', 'base_weight': 3.0},
        'account_level_reserve_large_or_increasing': {'dimension': 'Reserve/Hold', 'base_weight': 3.0},
        'deposit_total_non_positive': {'dimension': 'Cashflow', 'base_weight': 3.0},
        'long_gap_since_last_payout': {'dimension': 'Cashflow', 'base_weight': 1.0}
    }

    # Normalization parameters (example values, adjust based on data distribution)
    # For reserve_ratio: 0.0 to 1.0, higher is worse.
    # For payout_gap_days: 0 to 180+, higher is worse.
    # For negative_feedback_delta: 0 to 100 (percentage points), higher is worse.
    norm_params = {
        'reserve_ratio': {'min': 0.0, 'max': 0.5, 'scale': 1.0}, # 0.5 means 50% reserve ratio is max impact
        'payout_gap_days': {'min': 30, 'max': 180, 'scale': 1.0}, # 180 days is max impact
        'negative_feedback_delta': {'min': 0, 'max': 10, 'scale': 1.0} # 10 percentage points delta is max impact
    }

    def normalize_intensity(value, param_key):
        params = norm_params.get(param_key)
        if params is None or value is None:
            return 0.0

        min_val = params['min']
        max_val = params['max']
        scale = params['scale']

        if value <= min_val:
            return 0.0
        if value >= max_val:
            return scale

        return ((value - min_val) / (max_val - min_val)) * scale

    # Process signals and accumulate scores
    for signal in signals:
        signal_type = signal['type']
        if signal_type in signal_configs:
            config = signal_configs[signal_type]
            dimension = config['dimension']
            base_weight = config['base_weight']

            intensity = 1.0 # Default intensity if not normalized

            if signal_type == 'account_level_reserve_large_or_increasing' and signal['value'] and 'reserve_ratio' in signal['value']:
                intensity = normalize_intensity(signal['value']['reserve_ratio'], 'reserve_ratio')
            elif signal_type == 'long_gap_since_last_payout' and signal['value'] is not None:
                intensity = normalize_intensity(signal['value'], 'payout_gap_days')
            elif signal_type == 'negative_feedback_spike' and signal['delta'] is not None:
                intensity = normalize_intensity(signal['delta'], 'negative_feedback_delta')

            # For binary signals (forbidden_access, policy_warning_notifications, deposit_total_non_positive),
            # intensity is 1.0 if present, 0.0 if not.
            # For policy_compliance_increasing_with_action_required, severity can influence intensity.

            if signal_type == 'policy_compliance_increasing_with_action_required':
                if signal['severity'] == 'high':
                    intensity = 1.0
                elif signal['severity'] == 'medium':
                    intensity = 0.7
                else:
                    intensity = 0.3 # Low severity, less impact

            contribution = base_weight * intensity

            # Add contribution to dimension score, respecting the cap
            dimension_scores[dimension] = min(dimension_scores[dimension] + contribution, dimension_caps[dimension])

            signal_contributions.append({
                'type': signal_type,
                'contribution': contribution,
                'severity': signal['severity'],
                'evidence': signal['evidence']
            })

    # Calculate total risk score
    risk_score = sum(dimension_scores.values())

    # Escalation Logic:
    # If reserve is increasing AND policy warning/action required exists -> ensure risk_score >= 8
    reserve_increasing = any(s['type'] == 'account_level_reserve_large_or_increasing' and s['delta'] is not None and s['delta'] > 0 for s in signals)
    policy_warning_action_required = any(s['type'] == 'policy_warning_notifications' for s in signals) or \
                                     any(s['type'] == 'policy_compliance_increasing_with_action_required' and s['severity'] == 'high' for s in signals)

    if reserve_increasing and policy_warning_action_required:
        if risk_score < 8.0:
            risk_score = 8.0
            # Note: For v2, we might want to add a specific signal contribution for this escalation
            # to make it transparent in the top_signals, but for now, just adjust the score.

    # Determine risk_level
    if risk_score >= 8.0:
        risk_level = 'high'
    elif risk_score >= 5.0:
        risk_level = 'medium'
    else:
        risk_level = 'low'

    # Determine recommended_action
    if risk_level == 'high':
        recommended_action = 'lev1_review'
    elif risk_level == 'medium':
        recommended_action = 'lev2_review'
    else:
        recommended_action = 'none'

    # Get top 3 signals by contribution (descending)
    top_signals = sorted(signal_contributions, key=lambda x: x['contribution'], reverse=True)[:3]
    top_signals_summary = [f"{s['type']} (score +{s['contribution']:.2f})" for s in top_signals]

    return {
        'risk_score': risk_score,
        'risk_level': risk_level,
        'recommended_action': recommended_action,
        'top_signals': top_signals_summary
    }

In [ ]:
# prompt: Run scoring on the same 20 sellers and print a concise summary per seller.

print("--- Risk Scoring for 20 Random Sellers ---")
for index, row in random_sellers.iterrows():
    mp_sup_key = row['mp_sup_key']
    current_j = row['current_j']
    previous_j = row['previous_j']
    current_ts = row['current_ts']
    previous_ts = row['previous_ts']

    seller_signals = extract_signals(current_j, previous_j, current_ts, previous_ts)
    scoring_result = calculate_risk_score_v2(seller_signals)

    print(f"Seller: {mp_sup_key}")
    print(f"  Risk Score: {scoring_result['risk_score']}")
    print(f"  Risk Level: {scoring_result['risk_level']}")
    print(f"  Recommended Action: {scoring_result['recommended_action']}")
    print(f"  Top Signals: {'; '.join(scoring_result['top_signals']) if scoring_result['top_signals'] else 'None'}")

--- Risk Scoring for 20 Random Sellers ---
Seller: dc4bda0e-1737-4207-8b51-fc16e54920ee
  Risk Score: 0.0
  Risk Level: low
  Recommended Action: none
  Top Signals: None
Seller: fcaaf3dc-d0db-4605-9215-049562c3109b
  Risk Score: 2.0
  Risk Level: low
  Recommended Action: none
  Top Signals: policy_warning_notifications (score +2.00)
Seller: 2a1242c8-6a0d-4dec-8e5e-dc24dd1f6a12
  Risk Score: 0.0
  Risk Level: low
  Recommended Action: none
  Top Signals: None
Seller: 7aeea557-9f3a-4f4b-bec7-bbc71775e565
  Risk Score: 2.0
  Risk Level: low
  Recommended Action: none
  Top Signals: policy_warning_notifications (score +2.00)
Seller: adb7267c-813c-41bc-8968-fc951f11d585
  Risk Score: 0.0
  Risk Level: low
  Recommended Action: none
  Top Signals: None
Seller: a91a968f-0b57-4eda-9d19-7d6e15c8d96c
  Risk Score: 0.0
  Risk Level: low
  Recommended Action: none
  Top Signals: None
Seller: 90f1f44e-8de4-4560-934d-a095d0590e73
  Risk Score: 3.0
  Risk Level: low
  Recommended Action: none
  Top

# Generate Explanation Text

**Generate Explanation Text (Rule-Based First)**

In [ ]:
# prompt: Write a function generate_reason(risk_score, recommended_action, top_signals) that produces a 1–2 sentence summary:
# Requirements:
# - Use only the extracted evidence.
# - Do not invent facts.
# - Mention numeric evidence when available (e.g., reserve ratio, payout gap days, deposit total).
# - Keep it concise and readable for a risk analyst.
# - top_signals may be either:
#     (a) a list of signal dictionaries with keys like 'type', 'severity', 'evidence'
#     (b) a list of strings like "forbidden_access (score +3)"
# - If top_signals is a list of strings, extract the signal type from the string
#   and retrieve the corresponding evidence from the original extracted signals if available.
# - The function must handle both formats gracefully without errors.
# Generate reason for the 20 sample sellers and display them.

def generate_reason(risk_score, recommended_action, top_signals_summary, all_signals):
    """
    Generates a 1-2 sentence summary for the risk assessment.

    Args:
        risk_score (int): The calculated risk score.
        recommended_action (str): The recommended action.
        top_signals_summary (list): A list of strings like "signal_type (score +X)".
        all_signals (list): The complete list of signal dictionaries with 'type', 'severity', 'evidence'.

    Returns:
        str: A concise summary of the risk.
    """
    reason_parts = []

    if recommended_action == 'lev1_review':
        reason_parts.append(f"This seller requires a Level 1 review due to a high risk score of {risk_score}.")
    elif recommended_action == 'lev2_review':
        reason_parts.append(f"This seller requires a Level 2 review due to a medium risk score of {risk_score}.")
    else:
        reason_parts.append(f"This seller has a low risk score of {risk_score} and no immediate action is recommended.")

    if top_signals_summary:
        evidence_snippets = []
        for signal_summary_str in top_signals_summary:
            # Extract signal type from the summary string
            signal_type_match = re.match(r'(\w+) \(score \+\d+\)', signal_summary_str)
            if signal_type_match:
                signal_type = signal_type_match.group(1)
                # Find the corresponding signal in all_signals to get its evidence
                for signal in all_signals:
                    if signal['type'] == signal_type:
                        evidence_snippets.append(signal['evidence'])
                        break
            else:
                # Fallback if the format is not as expected, or if it's already a dict
                if isinstance(signal_summary_str, dict) and 'evidence' in signal_summary_str:
                    evidence_snippets.append(signal_summary_str['evidence'])
                else:
                    evidence_snippets.append(signal_summary_str) # Use as is if it's just a string

        if evidence_snippets:
            reason_parts.append("Key concerns include: " + "; ".join(evidence_snippets) + ".")

    return " ".join(reason_parts)

print("--- Risk Scoring and Reason Generation for 20 Random Sellers ---")
for index, row in random_sellers.iterrows():
    mp_sup_key = row['mp_sup_key']
    current_j = row['current_j']
    previous_j = row['previous_j']
    current_ts = row['current_ts']
    previous_ts = row['previous_ts']

    seller_signals = extract_signals(current_j, previous_j, current_ts, previous_ts)
    scoring_result = calculate_risk_score_v2(seller_signals)

    reason = generate_reason(
        scoring_result['risk_score'],
        scoring_result['recommended_action'],
        scoring_result['top_signals'],
        seller_signals # Pass all_signals to the function
    )

    print(f"Seller: {mp_sup_key}")
    print(f"  Risk Score: {scoring_result['risk_score']}")
    print(f"  Risk Level: {scoring_result['risk_level']}")
    print(f"  Recommended Action: {scoring_result['recommended_action']}")
    print(f"  Top Signals: {'; '.join(scoring_result['top_signals']) if scoring_result['top_signals'] else 'None'}")
    print(f"  Reason: {reason}")

--- Risk Scoring and Reason Generation for 20 Random Sellers ---
Seller: dc4bda0e-1737-4207-8b51-fc16e54920ee
  Risk Score: 0.0
  Risk Level: low
  Recommended Action: none
  Top Signals: None
  Reason: This seller has a low risk score of 0.0 and no immediate action is recommended.
Seller: fcaaf3dc-d0db-4605-9215-049562c3109b
  Risk Score: 2.0
  Risk Level: low
  Recommended Action: none
  Top Signals: policy_warning_notifications (score +2.00)
  Reason: This seller has a low risk score of 2.0 and no immediate action is recommended. Key concerns include: policy_warning_notifications (score +2.00).
Seller: 2a1242c8-6a0d-4dec-8e5e-dc24dd1f6a12
  Risk Score: 0.0
  Risk Level: low
  Recommended Action: none
  Top Signals: None
  Reason: This seller has a low risk score of 0.0 and no immediate action is recommended.
Seller: 7aeea557-9f3a-4f4b-bec7-bbc71775e565
  Risk Score: 2.0
  Risk Level: low
  Recommended Action: none
  Top Signals: policy_warning_notifications (score +2.00)
  Reason: T

(Optional) Vertex AI Rewrite (Polish Only)

In [ ]:
!pip install --upgrade google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.4/724.4 kB 20.0 MB/s eta 0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.61.0
    Uninstalling google-genai-1.61.0:
      Successfully uninstalled google-genai-1.61.0


# Final Output DataFrame + Top-K Alerts

**Final Output DataFrame + Top-K Alerts**

In [ ]:
# prompt: Create a final dataframe with one row per seller:
# Columns:
# - as_of_date (current_ts date)
# - mp_sup_key
# - current_ts
# - previous_ts
# - snapshot_gap_days
# - risk_score
# - risk_level
# - recommended_action
# - reason (use final_reason if available else rule-based reason)
# - signals_json (the full signals list)
# - agent_version ("json_agent_v1")
# Then:
# 1) Sort by risk_score descending.
# 2) Mark only the top 5 sellers as actionable (keep their recommended_action).
# 3) For all other sellers, set recommended_action = "none" but keep their score and signals.
# Print the top 10 sellers for review.

import pandas as pd
final_output_data = []

for index, row in merged_df.iterrows():
    mp_sup_key = row['mp_sup_key']
    current_j = row['current_j']
    previous_j = row['previous_j']
    current_ts = row['current_ts']
    previous_ts = row['previous_ts']
    snapshot_gap_days = row['snapshot_gap_days']

    seller_signals = extract_signals(current_j, previous_j, current_ts, previous_ts)
    scoring_result = calculate_risk_score_v2(seller_signals)

    reason = generate_reason(
        scoring_result['risk_score'],
        scoring_result['recommended_action'],
        scoring_result['top_signals'],
        seller_signals
    )

    final_output_data.append({
        'as_of_date': current_ts.date(),
        'mp_sup_key': mp_sup_key,
        'current_ts': current_ts,
        'previous_ts': previous_ts,
        'snapshot_gap_days': snapshot_gap_days,
        'risk_score': scoring_result['risk_score'],
        'risk_level': scoring_result['risk_level'],
        'recommended_action': scoring_result['recommended_action'],
        'reason': reason,
        'signals_json': json.dumps(seller_signals), # Store the full signals list as a JSON string
        'agent_version': "json_agent_v1"
    })

final_df = pd.DataFrame(final_output_data)

# 1) Sort by risk_score descending.
final_df = final_df.sort_values(by='risk_score', ascending=False).reset_index(drop=True)

# 2) Mark only the top 5 sellers as actionable (keep their recommended_action).
# 3) For all other sellers, set recommended_action = "none" but keep their score and signals.
if len(final_df) > 5:
    final_df.loc[5:, 'recommended_action'] = 'none'

print("Top 10 sellers for review:")
print(final_df.head(10))

Top 10 sellers for review:
   as_of_date                            mp_sup_key  \
0  2023-03-28  67a75c8a-2a0b-4428-8d29-0dbe0dbafe10   
1  2024-07-17  e4407d9b-da89-4267-8d78-1e5a67c76b65   
2  2024-03-08  ecb6ef14-fd13-4ed4-bc31-9e7674b34a27   
3  2022-06-29  c1c39cc8-5c4f-4aeb-8ac5-1366c8829d5a   
4  2022-01-09  9451bcff-e469-4269-85ff-4c2a6344006a   
5  2022-06-29  99a9b558-bc95-487a-887e-4666522d3e8a   
6  2023-04-04  7183a425-ce3a-4f08-a8c5-ad8e23fd6f78   
7  2022-06-29  5fa8b06b-c30f-48bb-8e8f-311826dee65e   
8  2023-11-24  29f920ee-0be4-48a7-8b5c-ad19d9572b6f   
9  2022-06-29  86a98251-6c1b-4f07-a2ee-1cd1703249b8   

                        current_ts                      previous_ts  \
0 2023-03-28 18:04:15.919697+00:00 2023-03-26 06:30:43.620999+00:00   
1 2024-07-17 20:06:33.310869+00:00 2022-08-31 16:57:17.578369+00:00   
2 2024-03-08 17:30:55.686073+00:00 2023-09-27 16:08:04.363031+00:00   
3 2022-06-29 15:10:20.979146+00:00 2022-06-28 15:35:52.399590+00:00   
4 2022-01-09

In [ ]:
# prompt: Print for 20 sellers:
# - compliance_score
# - cashflow_score
# - reserve_score
# - data_access_score
# - final risk_score

def calculate_risk_score_v2_with_dimensions(signals):
    """
    Scores a list of signals using float contributions and normalized intensity,
    and returns individual dimension scores.
    """
    dimension_scores = {
        'Data Access': 0.0,
        'Compliance': 0.0,
        'Cashflow': 0.0,
        'Reserve/Hold': 0.0
    }
    dimension_caps = {
        'Data Access': 3.0,
        'Compliance': 4.0,
        'Cashflow': 4.0,
        'Reserve/Hold': 4.0
    }
    signal_contributions = []

    # Define signal base weights and dimension mapping
    signal_configs = {
        'forbidden_access': {'dimension': 'Data Access', 'base_weight': 3.0},
        'policy_warning_notifications': {'dimension': 'Compliance', 'base_weight': 2.0},
        'negative_feedback_spike': {'dimension': 'Compliance', 'base_weight': 1.0},
        'policy_compliance_increasing_with_action_required': {'dimension': 'Compliance', 'base_weight': 3.0},
        'account_level_reserve_large_or_increasing': {'dimension': 'Reserve/Hold', 'base_weight': 3.0},
        'deposit_total_non_positive': {'dimension': 'Cashflow', 'base_weight': 3.0},
        'long_gap_since_last_payout': {'dimension': 'Cashflow', 'base_weight': 1.0}
    }

    # Normalization parameters (example values, adjust based on data distribution)
    norm_params = {
        'reserve_ratio': {'min': 0.0, 'max': 0.5, 'scale': 1.0},
        'payout_gap_days': {'min': 30, 'max': 180, 'scale': 1.0},
        'negative_feedback_delta': {'min': 0, 'max': 10, 'scale': 1.0}
    }

    def normalize_intensity(value, param_key):
        params = norm_params.get(param_key)
        if params is None or value is None:
            return 0.0

        min_val = params['min']
        max_val = params['max']
        scale = params['scale']

        if value <= min_val:
            return 0.0
        if value >= max_val:
            return scale

        return ((value - min_val) / (max_val - min_val)) * scale

    # Process signals and accumulate scores
    for signal in signals:
        signal_type = signal['type']
        if signal_type in signal_configs:
            config = signal_configs[signal_type]
            dimension = config['dimension']
            base_weight = config['base_weight']

            intensity = 1.0 # Default intensity if not normalized

            if signal_type == 'account_level_reserve_large_or_increasing' and signal['value'] and 'reserve_ratio' in signal['value']:
                intensity = normalize_intensity(signal['value']['reserve_ratio'], 'reserve_ratio')
            elif signal_type == 'long_gap_since_last_payout' and signal['value'] is not None:
                intensity = normalize_intensity(signal['value'], 'payout_gap_days')
            elif signal_type == 'negative_feedback_spike' and signal['delta'] is not None:
                intensity = normalize_intensity(signal['delta'], 'negative_feedback_delta')

            if signal_type == 'policy_compliance_increasing_with_action_required':
                if signal['severity'] == 'high':
                    intensity = 1.0
                elif signal['severity'] == 'medium':
                    intensity = 0.7
                else:
                    intensity = 0.3

            contribution = base_weight * intensity

            dimension_scores[dimension] = min(dimension_scores[dimension] + contribution, dimension_caps[dimension])

            signal_contributions.append({
                'type': signal_type,
                'contribution': contribution,
                'severity': signal['severity'],
                'evidence': signal['evidence']
            })

    # Calculate total risk score

    risk_score = sum(dimension_scores.values())

    # Escalation Logic:
    reserve_increasing = any(s['type'] == 'account_level_reserve_large_or_increasing' and s['delta'] is not None and s['delta'] > 0 for s in signals)
    policy_warning_action_required = any(s['type'] == 'policy_warning_notifications' for s in signals) or \
                                     any(s['type'] == 'policy_compliance_increasing_with_action_required' and s['severity'] == 'high' for s in signals)

    if reserve_increasing and policy_warning_action_required:
        if risk_score < 8.0:
            risk_score = 8.0

    # Determine risk_level
    if risk_score >= 8.0:
        risk_level = 'high'
    elif risk_score >= 5.0:
        risk_level = 'medium'
    else:
        risk_level = 'low'

    # Determine recommended_action
    if risk_level == 'high':
        recommended_action = 'lev1_review'
    elif risk_level == 'medium':
        recommended_action = 'lev2_review'
    else:
        recommended_action = 'none'

    # Get top 3 signals by contribution (descending)
    top_signals = sorted(signal_contributions, key=lambda x: x['contribution'], reverse=True)[:3]
    top_signals_summary = [f"{s['type']} (score +{s['contribution']:.2f})" for s in top_signals]

    return {
        'risk_score': risk_score,
        'risk_level': risk_level,
        'recommended_action': recommended_action,
        'top_signals': top_signals_summary,
        'dimension_scores': dimension_scores # Return dimension scores
    }

In [ ]:
# prompt: print them out

import pandas as pd
final_output_data_v2 = []

for index, row in merged_df.iterrows():
    mp_sup_key = row['mp_sup_key']
    current_j = row['current_j']
    previous_j = row['previous_j']
    current_ts = row['current_ts']
    previous_ts = row['previous_ts']
    snapshot_gap_days = row['snapshot_gap_days']

    seller_signals = extract_signals(current_j, previous_j, current_ts, previous_ts)
    scoring_result_v2 = calculate_risk_score_v2_with_dimensions(seller_signals)

    reason = generate_reason(
        scoring_result_v2['risk_score'],
        scoring_result_v2['recommended_action'],
        scoring_result_v2['top_signals'],
        seller_signals
    )

    final_output_data_v2.append({
        'as_of_date': current_ts.date(),
        'mp_sup_key': mp_sup_key,
        'current_ts': current_ts,
        'previous_ts': previous_ts,
        'snapshot_gap_days': snapshot_gap_days,
        'risk_score': scoring_result_v2['risk_score'],
        'risk_level': scoring_result_v2['risk_level'],
        'recommended_action': scoring_result_v2['recommended_action'],
        'reason': reason,
        'signals_json': json.dumps(seller_signals),
        'agent_version': "json_agent_v2",
        'data_access_score': scoring_result_v2['dimension_scores']['Data Access'],
        'compliance_score': scoring_result_v2['dimension_scores']['Compliance'],
        'cashflow_score': scoring_result_v2['dimension_scores']['Cashflow'],
        'reserve_hold_score': scoring_result_v2['dimension_scores']['Reserve/Hold']
    })

final_df_v2 = pd.DataFrame(final_output_data_v2)

# Sort by risk_score descending.
final_df_v2 = final_df_v2.sort_values(by='risk_score', ascending=False).reset_index(drop=True)

# Mark only the top 5 sellers as actionable (keep their recommended_action).
# For all other sellers, set recommended_action = "none" but keep their score and signals.
if len(final_df_v2) > 5:
    final_df_v2.loc[5:, 'recommended_action'] = 'none'

print("Final Output DataFrame (v2) with Dimension Scores:")
print(final_df_v2.head(10))

Final Output DataFrame (v2) with Dimension Scores:
   as_of_date                            mp_sup_key  \
0  2023-03-28  67a75c8a-2a0b-4428-8d29-0dbe0dbafe10   
1  2024-07-17  e4407d9b-da89-4267-8d78-1e5a67c76b65   
2  2024-03-08  ecb6ef14-fd13-4ed4-bc31-9e7674b34a27   
3  2022-06-29  c1c39cc8-5c4f-4aeb-8ac5-1366c8829d5a   
4  2022-01-09  9451bcff-e469-4269-85ff-4c2a6344006a   
5  2022-06-29  99a9b558-bc95-487a-887e-4666522d3e8a   
6  2023-04-04  7183a425-ce3a-4f08-a8c5-ad8e23fd6f78   
7  2022-06-29  5fa8b06b-c30f-48bb-8e8f-311826dee65e   
8  2023-11-24  29f920ee-0be4-48a7-8b5c-ad19d9572b6f   
9  2022-06-29  86a98251-6c1b-4f07-a2ee-1cd1703249b8   

                        current_ts                      previous_ts  \
0 2023-03-28 18:04:15.919697+00:00 2023-03-26 06:30:43.620999+00:00   
1 2024-07-17 20:06:33.310869+00:00 2022-08-31 16:57:17.578369+00:00   
2 2024-03-08 17:30:55.686073+00:00 2023-09-27 16:08:04.363031+00:00   
3 2022-06-29 15:10:20.979146+00:00 2022-06-28 15:35:52.3995

# Store output to supabase

In [ ]:
import os

os.environ["SUPABASE_SERVICE_ROLE_KEY"] = "your_supabase_service_role_key_here"

In [ ]:
# prompt: # prompt:
# I cannot connect to Supabase Postgres via port 5432 because my environment has no IPv6 egress (Supabase free plan is IPv6-only).
# Generate Python code to upload a pandas DataFrame named `final_df_v2` into Supabase using the HTTPS REST API (PostgREST) instead.
# My Supabase Project URL is:
# https://your-project.supabase.co
# Requirements:
# - Use `requests` (no supabase-py dependency).
# - Use `SUPABASE_SERVICE_ROLE_KEY` from environment variables (raise a clear error if not set).
# - Create a helper function `ensure_table_exists(table_name, df)` that generates and prints a SQL `CREATE TABLE IF NOT EXISTS` statement based on `final_df_v2.dtypes` (text, integer, double precision, boolean, timestamptz). Note: do NOT execute SQL via DB connection (only print it so I can run it in Supabase SQL Editor).
# - Upload rows via POST to `{SUPABASE_URL}/rest/v1/{table_name}` with headers:
#   - apikey: service role key
#   - Authorization: Bearer service role key
#   - Content-Type: application/json
#   - Prefer: return=minimal
# - Upload in batches (e.g., 500 rows per request) to avoid payload limits.
# - Handle NaN/NaT by converting them to None before JSON serialization.
# - After each batch, check status_code; if >= 300, print status + response.text and stop.
# - At the end, print a success message with how many rows uploaded.
# - Include an optional `upsert` mode using `Prefer: resolution=merge-duplicates` and `on_conflict` parameter (assume a primary key column exists named `mp_sup_key`; if it doesn't exist, fall back to pure insert and warn).
# - Provide an example call that uploads `final_df_v2` to table `final_output_v2`.
# Output:
# - Output only the final Python code block, ready to run in a notebook.

import os
import json
import requests
import numpy as np
import pandas as pd
import datetime


def ensure_table_exists(table_name: str, df: pd.DataFrame) -> None:
    """
    Generates and prints a SQL CREATE TABLE IF NOT EXISTS statement based on DataFrame dtypes.
    NOTE: This does NOT execute SQL. Copy/paste into Supabase SQL Editor.
    """
    type_mapping = {
        "object": "TEXT",
        "string": "TEXT",
        "category": "TEXT",
        "bool": "BOOLEAN",
        "int64": "BIGINT",
        "int32": "INTEGER",
        "float64": "DOUBLE PRECISION",
        "float32": "REAL",
        "datetime64[ns]": "TIMESTAMPTZ",
        "datetime64[ns, UTC]": "TIMESTAMPTZ",
    }

    columns_sql = []
    for col_name, dtype in df.dtypes.items():
        sql_type = type_mapping.get(str(dtype), "TEXT")

        # If it's an object column, it might actually contain date/datetime objects
        if pd.api.types.is_object_dtype(dtype):
            non_null = df[col_name].dropna()
            if len(non_null) > 0:
                first_val = non_null.iloc[0]
                if isinstance(first_val, (pd.Timestamp, datetime.datetime, datetime.date)):
                    # date/datetime -> use DATE if it's date-only, otherwise TIMESTAMPTZ
                    if isinstance(first_val, datetime.date) and not isinstance(first_val, datetime.datetime):
                        sql_type = "DATE"
                    else:
                        sql_type = "TIMESTAMPTZ"

        if col_name == "mp_sup_key":
            columns_sql.append(f'"{col_name}" {sql_type} PRIMARY KEY')
        else:
            columns_sql.append(f'"{col_name}" {sql_type}')

    create_table_sql = f'''
CREATE TABLE IF NOT EXISTS "{table_name}" (
  {", ".join(columns_sql)}
);
'''.strip()

    print("Run this in Supabase SQL Editor:")
    print(create_table_sql)


def df_to_json_records(df: pd.DataFrame) -> list[dict]:
    """
    Convert a DataFrame batch to JSON-serializable list[dict]:
    - datetime/Timestamp/date -> ISO strings
    - NaN/NaT/pd.NA -> None
    """
    out = df.copy()

    for col in out.columns:
        s = out[col]

        # datetime64 columns -> ISO strings
        if pd.api.types.is_datetime64_any_dtype(s):
            out[col] = s.apply(lambda x: x.isoformat() if pd.notna(x) else None)
            continue

        # object columns might contain datetime/date/Timestamp
        if pd.api.types.is_object_dtype(s):
            non_null = s.dropna()
            if len(non_null) > 0:
                first = non_null.iloc[0]
                if isinstance(first, (pd.Timestamp, datetime.datetime, datetime.date)):
                    out[col] = s.apply(lambda x: x.isoformat() if pd.notna(x) else None)

    # Replace NaN/NaT/pd.NA with None
    out = out.where(pd.notna(out), None)

    return out.to_dict(orient="records")


def upload_dataframe_to_supabase(
    df: pd.DataFrame,
    table_name: str,
    supabase_url: str,
    upsert: bool = False,
    batch_size: int = 500,
) -> None:
    """
    Upload a pandas DataFrame to Supabase via PostgREST HTTPS API.

    - Uses SUPABASE_SERVICE_ROLE_KEY from environment
    - Batch uploads to avoid payload limits
    - Optional upsert on mp_sup_key (if present)
    """
    supabase_key = os.environ.get("SUPABASE_SERVICE_ROLE_KEY")
    if not supabase_key:
        raise ValueError("SUPABASE_SERVICE_ROLE_KEY environment variable not set.")

    headers = {
        "apikey": supabase_key,
        "Authorization": f"Bearer {supabase_key}",
        "Content-Type": "application/json",
        "Prefer": "return=minimal",
    }

    # Determine URL (insert vs upsert)
    if upsert:
        if "mp_sup_key" not in df.columns:
            print("Warning: upsert=True but 'mp_sup_key' column not found. Falling back to pure insert.")
            upsert = False
        else:
            headers["Prefer"] = "resolution=merge-duplicates,return=minimal"
            url = f"{supabase_url}/rest/v1/{table_name}?on_conflict=mp_sup_key"
    if not upsert:
        url = f"{supabase_url}/rest/v1/{table_name}"

    total_rows_uploaded = 0
    num_batches = (len(df) + batch_size - 1) // batch_size

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(df))
        batch_df = df.iloc[start_idx:end_idx]

        payload = df_to_json_records(batch_df)

        resp = requests.post(url, headers=headers, json=payload)
        if resp.status_code >= 300:
            print(f"Error uploading batch {i+1}/{num_batches}")
            print("Status:", resp.status_code)
            print("Body:", resp.text)
            break

        total_rows_uploaded += len(batch_df)
        print(f"Batch {i+1}/{num_batches} uploaded. Total rows uploaded: {total_rows_uploaded}")

    print(f"Finished. Total rows uploaded: {total_rows_uploaded}")


# -------------------------
# Example usage (your case)
# -------------------------
supabase_url = "https://your-project.supabase.co"
table_name = "risk_scores"

# 1) Print CREATE TABLE SQL (copy/paste into Supabase SQL Editor)
ensure_table_exists(table_name, final_df_v2)

# 2) Upload (requires env var SUPABASE_SERVICE_ROLE_KEY set)
upload_dataframe_to_supabase(final_df_v2, table_name, supabase_url, upsert=True, batch_size=500)

Run this in Supabase SQL Editor:
CREATE TABLE IF NOT EXISTS "risk_scores" (
  "as_of_date" DATE, "mp_sup_key" TEXT PRIMARY KEY, "current_ts" TIMESTAMPTZ, "previous_ts" TIMESTAMPTZ, "snapshot_gap_days" DOUBLE PRECISION, "risk_score" DOUBLE PRECISION, "risk_level" TEXT, "recommended_action" TEXT, "reason" TEXT, "signals_json" TEXT, "agent_version" TEXT, "data_access_score" DOUBLE PRECISION, "compliance_score" DOUBLE PRECISION, "cashflow_score" DOUBLE PRECISION, "reserve_hold_score" DOUBLE PRECISION
);
Batch 1/2 uploaded. Total rows uploaded: 500
Batch 2/2 uploaded. Total rows uploaded: 1000
Finished. Total rows uploaded: 1000
